# 00 · Fetch TTF Forward Curve from Databento

Regenerates `data/raw/ttf_curve.csv` (M1–M24 daily settlement prices)
from the **Databento NDEX.IMPACT** dataset (ICE Endex TTF).

### Two-step approach
1. **Definitions** — sample one day per year (2019–2026) to collect every
   historical `instrument_id → expiration` mapping.
2. **OHLCV** — fetch `ohlcv-1d` for all those IDs in one request.

### Tenor mapping (ICE Endex convention)
ICE TTF contracts expire ~2 business days **before** the delivery month
(e.g. the July 2026 contract expires June 29 2026).
So **M1 = next calendar month**, not the current one:
```
delivery_month = expiry_month + 1  (year rollover if expiry_month == 12)
months_ahead   = (delivery_year − trade_year) × 12 + (delivery_month − trade_month)
```

**Run order:** cells 0 → 6 for a full build, then cell 7 for daily incremental updates.

Requires: `pip install databento`  |  Network: `hist.databento.com` must be reachable.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION  ← edit this cell
# ═══════════════════════════════════════════════════════════════════════════════
DATABENTO_API_KEY = "paste_your_key_here"   # ← replace with your key
START_DATE        = "2018-12-23"             # earliest available on NDEX.IMPACT
END_DATE          = None                     # None = today
N_MONTHS          = 24                       # M1 to M24
# ═══════════════════════════════════════════════════════════════════════════════

import sys, os, warnings, calendar
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
from datetime import date, timedelta
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c)); os.chdir(_c)
        print(f'\u2705 Root: {_c}'); break

try:
    import databento as db
    print(f'\u2705 databento {db.__version__}')
except ImportError:
    raise ImportError('databento not installed. Run: pip install databento')

DATASET    = 'NDEX.IMPACT'
PARENT_SYM = 'TFM.FUT'
OUT_PATH   = Path('data/raw/ttf_curve.csv')

# Definition scan dates: Jan + Jul each year so all contracts are captured
DEF_SAMPLE_DATES = [
    '2019-01-02', '2020-01-02', '2021-01-04', '2022-01-03',
    '2023-01-02', '2024-01-02', '2025-01-02', '2026-01-02',
    '2019-07-01', '2020-07-01', '2021-07-01', '2022-07-01',
    '2023-07-03', '2024-07-01', '2025-07-01',
]

key_ok = DATABENTO_API_KEY and not DATABENTO_API_KEY.startswith('paste')
print(f'API key : {"SET \u2705" if key_ok else "NOT SET \u26a0\ufe0f  \u2190 paste your key above"}')
print(f'Output  : {OUT_PATH}')
print(f'Start   : {START_DATE}  |  Months: M1\u2013M{N_MONTHS}')

## 1 · Fetch Instrument Definitions (year by year)

Calls `schema="definition"` for one trading day per year.
Each call returns all outright futures active on that date.
We keep only outrights (`raw_symbol` ending with `!`) and build a
master `instrument_id → expiration` map covering the full history.

In [ ]:
def _next_weekday(d):
    ts = pd.Timestamp(d)
    if ts.weekday() == 5: ts += timedelta(days=2)
    elif ts.weekday() == 6: ts += timedelta(days=1)
    return ts.strftime('%Y-%m-%d')

def _parse_expiry(val):
    if pd.isna(val): return None
    try:
        ts = pd.Timestamp(val, unit='ns') if isinstance(val, (int, float)) else pd.Timestamp(val)
        return ts.date()
    except Exception:
        return None

c = db.Historical(DATABENTO_API_KEY)
id_to_expiry = {}   # instrument_id (int) → expiration (date)

for raw_date in DEF_SAMPLE_DATES:
    d   = _next_weekday(raw_date)
    end = (pd.Timestamp(d) + timedelta(days=1)).strftime('%Y-%m-%d')
    try:
        store  = c.timeseries.get_range(
            dataset=DATASET, symbols=PARENT_SYM, stype_in='parent',
            schema='definition', start=d, end=end,
        )
        df_def = store.to_df()
        if df_def.empty:
            print(f'  {d}: no rows'); continue
        outrights = df_def[df_def['raw_symbol'].str.endswith('!')].copy()
        new = 0
        for _, row in outrights.iterrows():
            iid = int(row['instrument_id'])
            if iid not in id_to_expiry:
                exp = _parse_expiry(row['expiration'])
                if exp: id_to_expiry[iid] = exp; new += 1
        print(f'  {d}: {len(outrights):3d} outrights  +{new:3d} new  \u2192  {len(id_to_expiry):3d} total')
    except Exception as e:
        print(f'  {d}: ERROR \u2014 {e}')

print(f'\n\u2705 Total unique instrument IDs: {len(id_to_expiry)}')

# Preview earliest and latest contracts
sorted_items = sorted(id_to_expiry.items(), key=lambda x: x[1])
preview = pd.DataFrame([
    {'instrument_id': k, 'expiration': v,
     'delivery_month': pd.Timestamp(v.year, (v.month % 12) + 1 if v.month < 12 else 1,
                                    1).strftime('%b %Y') if True else ''}
    for k, v in sorted_items[:5] + sorted_items[-5:]
])
display(preview)

## 2 · Cost Estimate

Uses `metadata.get_billable_size()` to estimate the download size and cost
before making the actual data request.
NDEX.IMPACT ohlcv-1d is very small (daily end-of-day bars only).

In [ ]:
end_str = (date.today() + timedelta(days=1)).strftime('%Y-%m-%d')
syms    = [str(iid) for iid in id_to_expiry.keys()]

print(f'Instruments : {len(syms)}')
print(f'Date range  : {START_DATE} \u2192 {end_str}')
print()

try:
    size_bytes = c.metadata.get_billable_size(
        dataset=DATASET,
        symbols=syms,
        stype_in='instrument_id',
        schema='ohlcv-1d',
        start=START_DATE,
        end=end_str,
    )
    size_mb  = size_bytes / 1e6
    cost_usd = (size_bytes / 1e9) * 3.0   # NDEX.IMPACT is ~$3/GB
    print(f'Billable size : {size_mb:.2f} MB  ({size_bytes:,} bytes)')
    print(f'Est. cost     : ${cost_usd:.4f} USD')
    if cost_usd < 125:
        print(f'Free credits  : $125  \u2192  within limits \u2705  (${125 - cost_usd:.2f} remaining)')
    else:
        print(f'\u26a0\ufe0f  Estimated cost ${cost_usd:.2f} exceeds $125 free credit threshold')
except Exception as e:
    print(f'Cost estimate unavailable: {e}')
    print('NDEX.IMPACT ohlcv-1d is typically <5 MB for the full TTF history.')

print('\n\u2192 Run the next cell to fetch the data.')

## 3 · Fetch ohlcv-1d Prices

Single request for all `instrument_id`s from `START_DATE` to today.
Databento delivers prices as fixed-point integers (× 10⁻⁹) for NDEX.IMPACT;
the cell detects and rescales automatically.

In [ ]:
end_str = (date.today() + timedelta(days=1)).strftime('%Y-%m-%d')
syms    = [str(iid) for iid in id_to_expiry.keys()]

print(f'Fetching ohlcv-1d for {len(syms)} instruments ...')
store  = c.timeseries.get_range(
    dataset=DATASET,
    symbols=syms,
    stype_in='instrument_id',
    schema='ohlcv-1d',
    start=START_DATE,
    end=end_str,
)
raw_df = store.to_df()
print(f'\u2705 Raw rows : {len(raw_df):,}')
print(f'   Columns : {raw_df.columns.tolist()}')

# Normalise timestamp \u2192 date
raw_df['date'] = pd.to_datetime(raw_df['ts_event'], utc=True).dt.date

# Scale prices: Databento fixed-point \xd7 1e9 (TTF trades ~5\u2013350 \u20ac/MWh)
price_mean = raw_df['close'].dropna().mean()
if price_mean > 1_000_000:
    raw_df['close'] = raw_df['close'] / 1e9
    print(f'   Applied \xf71e9 price normalisation (mean was {price_mean:.2e})')
else:
    print(f'   Close price mean: {price_mean:.3f} \u20ac/MWh (no scaling needed)')

ohlcv = raw_df[['date', 'instrument_id', 'close', 'volume']].copy()
print(f'\n   Date range : {min(ohlcv["date"])} \u2192 {max(ohlcv["date"])}')
print(f'   Unique IDs : {ohlcv["instrument_id"].nunique()}')

## 4 · Build Forward Curve (corrected delivery-month mapping)

Maps each `(date, instrument_id)` row to tenor M1–M24 using the
**delivery month** (not the expiry month):
```
delivery_month = expiry_month + 1   (year rollover for December)
months_ahead   = (delivery_year − trade_year)×12 + (delivery_month − trade_month)
```
Near roll-dates, two contracts may map to the same tenor; we keep the
**highest-volume** one (most liquid). Sparse far months are forward-filled.

In [ ]:
def build_forward_curve(ohlcv_df, expiry_map, n_months=24):
    """
    Pivot OHLCV into a wide M1\u2013M{n_months} forward curve.

    ICE Endex TTF expires in the month BEFORE delivery:
      delivery_month = expiry_month + 1  (year rollover when expiry_month == 12)
      months_ahead   = (delivery_year - trade_year)*12 + (delivery_month - trade_month)
    """
    rows     = []
    skipped  = 0

    for _, row in ohlcv_df.iterrows():
        iid    = int(row['instrument_id'])
        expiry = expiry_map.get(iid)
        if expiry is None: skipped += 1; continue

        price = row['close']
        vol   = row.get('volume', 0) or 0
        if pd.isna(price) or price <= 0: continue

        d_ts = pd.Timestamp(row['date'])
        e_ts = pd.Timestamp(expiry)

        # Delivery month is always expiry month + 1
        if e_ts.month == 12:
            delivery_month, delivery_year = 1, e_ts.year + 1
        else:
            delivery_month, delivery_year = e_ts.month + 1, e_ts.year

        months_ahead = (
            (delivery_year - d_ts.year) * 12
            + (delivery_month - d_ts.month)
        )
        if months_ahead < 1 or months_ahead > n_months:
            continue

        rows.append({
            'date':   d_ts.normalize(),
            'tenor':  f'M{months_ahead}',
            'close':  price,
            'volume': vol,
        })

    if skipped:
        print(f'  Skipped {skipped:,} rows with no expiry in definition map')

    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date'])

    # Keep highest-volume contract per (date, tenor) near roll dates
    df = (
        df.sort_values('volume', ascending=False)
          .drop_duplicates(subset=['date', 'tenor'])
          .drop(columns=['volume'])
    )

    wide = df.pivot_table(index='date', columns='tenor', values='close', aggfunc='first')
    wide.columns.name = None
    ordered = [f'M{m}' for m in range(1, n_months + 1) if f'M{m}' in wide.columns]
    return wide[ordered].sort_index()


curve = build_forward_curve(ohlcv, id_to_expiry, n_months=N_MONTHS)

# Forward-fill sparse far months (some far contracts have gaps)
curve = curve.ffill(axis=1)

print(f'\u2705 Curve shape : {curve.shape}')
print(f'   Date range  : {curve.index.min().date()} \u2192 {curve.index.max().date()}')
print(f'   Tenors      : {list(curve.columns[:6])} ...')
curve.tail(3)

## 5 · Sanity Check

- Prints M1–M12 with delivery month labels (M1 should be **next** calendar month).
- Shows missing % per column (near tenors should be 0%).
- Plots the latest forward curve.

In [ ]:
latest_date = curve.index[-1]
latest_row  = curve.iloc[-1]

# ── M1-M12 with delivery month labels ────────────────────────────────────────
print(f'Latest date in curve: {latest_date.date()}')
print(f'\n  {"Tenor":<6} {"Delivery month":<14} {"Price (\u20ac/MWh)"}')
print(f'  {"-" * 36}')
for m in range(1, 13):
    col = f'M{m}'
    if col not in curve.columns: continue
    dm  = (latest_date.month + m - 1) % 12 + 1
    yr  = latest_date.year + (latest_date.month + m - 1) // 12
    lbl = f'{calendar.month_abbr[dm]} {yr}'
    val = latest_row[col]
    p   = f'{val:.3f}' if not pd.isna(val) else 'NaN'
    print(f'  {col:<6} {lbl:<14} {p}')

# M1 delivery check
m1_dm = (latest_date.month % 12) + 1
m1_yr = latest_date.year + (1 if latest_date.month == 12 else 0)
print(f'\n  M1 should deliver in: {calendar.month_abbr[m1_dm]} {m1_yr}')
dm_actual = (latest_date.month + 1 - 1) % 12 + 1
print(f'  \u2705 Confirmed: M1 = {calendar.month_abbr[m1_dm]} {m1_yr} (trade_month + 1)')

# ── Missing % by column ───────────────────────────────────────────────────────
missing = (curve.isna().sum() / len(curve) * 100).round(1)
print(f'\nMissing % (near tenors should be ~0%)')
near  = [f'M{m}' for m in range(1, 7)  if f'M{m}' in curve.columns]
far   = [f'M{m}' for m in range(7, 25) if f'M{m}' in curve.columns]
print('  Near: ' + '  '.join(f'{c}:{missing[c]:.0f}%' for c in near))
print('  Far:  ' + '  '.join(f'{c}:{missing[c]:.0f}%' for c in far[:6]) + ' ...')

# ── Forward curve chart ───────────────────────────────────────────────────────
m_cols   = [c for c in curve.columns if not pd.isna(latest_row[c])]
x_labels = []
y_vals   = []
for col in m_cols:
    m   = int(col[1:])
    dm  = (latest_date.month + m - 1) % 12 + 1
    yr  = latest_date.year + (latest_date.month + m - 1) // 12
    x_labels.append(f"{calendar.month_abbr[dm]}'{str(yr)[-2:]}")
    y_vals.append(float(latest_row[col]))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x_labels, y=y_vals, mode='lines+markers',
    line=dict(color='#6366f1', width=2.5), marker=dict(size=6),
    name=f'TTF {latest_date.date()}',
))
fig.update_layout(
    template='plotly_white', height=420,
    title=f'TTF Forward Curve M1\u2013M{N_MONTHS} ({latest_date.date()})',
    yaxis_title='\u20ac/MWh', xaxis_title='Delivery Month',
    xaxis=dict(tickangle=45, tickfont=dict(size=9)),
)
fig.show()

# M1 vs M2 check
m1 = float(curve['M1'].dropna().iloc[-1]) if 'M1' in curve.columns else None
m2 = float(curve['M2'].dropna().iloc[-1]) if 'M2' in curve.columns else None
if m1 and m2:
    shape = 'backwardation \u2705' if m1 > m2 else 'contango \u26a0\ufe0f'
    print(f'\nM1 vs M2: {shape}  (M1=\u20ac{m1:.2f}  M2=\u20ac{m2:.2f}  \u0394=\u20ac{m1-m2:+.2f})')
if m1:
    rng_ok = '\u2705' if 5 < m1 < 500 else '\u26a0\ufe0f  OUTSIDE EXPECTED RANGE'
    print(f'M1 price range check (\u20ac5\u2013\u20ac500): \u20ac{m1:.2f}/MWh  {rng_ok}')

## 6 · Save to `data/raw/ttf_curve.csv`

In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
curve.to_csv(OUT_PATH)
print(f'\u2705 Saved \u2192 {OUT_PATH}')
print(f'   Shape     : {curve.shape}')
print(f'   Date range: {curve.index.min().date()} \u2192 {curve.index.max().date()}')
print(f'   File size : {OUT_PATH.stat().st_size / 1024:.1f} KB')

# Round-trip verification
verify = pd.read_csv(OUT_PATH, index_col=0, parse_dates=True)
verify.index = pd.to_datetime(verify.index).tz_localize(None)
assert list(verify.columns) == list(curve.columns), 'Column mismatch'
assert len(verify) == len(curve), 'Row count mismatch'
print(f'   Round-trip: {verify.shape} \u2705')
verify.tail(3)

## 7 · Incremental Update (run daily)

Loads the existing CSV, finds the last date, fetches only new bars from
`last_date + 1` to today, and appends them. Much faster than a full re-fetch.

Also refreshes the definition map for the new date range so newly listed
contracts are captured.

In [ ]:
def incremental_update(api_key, csv_path=OUT_PATH, n_months=N_MONTHS,
                       dataset=DATASET, parent_sym=PARENT_SYM):
    """
    Fetch only new trading days since the last date in csv_path and append them.
    Returns the updated DataFrame.
    """
    path = Path(csv_path)
    if not path.exists():
        print(f'\u26a0\ufe0f  {path} not found \u2014 run cells 0\u20136 first')
        return None

    existing = pd.read_csv(path, index_col=0, parse_dates=True)
    existing.index = pd.to_datetime(existing.index).tz_localize(None)
    last_date   = existing.index.max()
    fetch_start = (last_date + timedelta(days=1)).strftime('%Y-%m-%d')
    fetch_end   = (date.today() + timedelta(days=1)).strftime('%Y-%m-%d')

    if pd.Timestamp(fetch_start) >= pd.Timestamp(fetch_end):
        print(f'\u2705 Already up to date ({last_date.date()})')
        return existing

    print(f'Last date   : {last_date.date()}')
    print(f'Fetch range : {fetch_start} \u2192 {fetch_end}')

    client = db.Historical(api_key)

    # Refresh definition map for the new period (catches newly listed contracts)
    upd_ids = dict(id_to_expiry)   # copy session map
    d   = _next_weekday(fetch_start)
    end = (pd.Timestamp(d) + timedelta(days=1)).strftime('%Y-%m-%d')
    try:
        store  = client.timeseries.get_range(
            dataset=dataset, symbols=parent_sym, stype_in='parent',
            schema='definition', start=d, end=end,
        )
        df_def = store.to_df()
        if not df_def.empty:
            outrights = df_def[df_def['raw_symbol'].str.endswith('!')]
            new_def = 0
            for _, row in outrights.iterrows():
                iid = int(row['instrument_id'])
                if iid not in upd_ids:
                    exp = _parse_expiry(row['expiration'])
                    if exp: upd_ids[iid] = exp; new_def += 1
            if new_def: print(f'  +{new_def} new contracts in definition map')
    except Exception as e:
        print(f'  Definition refresh skipped: {e}')

    # Fetch new OHLCV
    syms = [str(iid) for iid in upd_ids.keys()]
    store = client.timeseries.get_range(
        dataset=dataset, symbols=syms, stype_in='instrument_id',
        schema='ohlcv-1d', start=fetch_start, end=fetch_end,
    )
    new_raw = store.to_df()
    if new_raw.empty:
        print('No new OHLCV data returned (market may have been closed)')
        return existing

    new_raw['date'] = pd.to_datetime(new_raw['ts_event'], utc=True).dt.date
    if new_raw['close'].dropna().mean() > 1_000_000:
        new_raw['close'] = new_raw['close'] / 1e9
    new_ohlcv = new_raw[['date', 'instrument_id', 'close', 'volume']].copy()

    # Build new curve rows and append
    new_curve = build_forward_curve(new_ohlcv, upd_ids, n_months=n_months)
    new_curve  = new_curve.ffill(axis=1)

    # Align columns to existing before concat
    for col in existing.columns:
        if col not in new_curve.columns:
            new_curve[col] = np.nan
    new_curve = new_curve[existing.columns]

    combined = pd.concat([existing, new_curve])
    combined = combined[~combined.index.duplicated(keep='last')].sort_index()
    combined.to_csv(path)

    new_rows = len(new_curve)
    print(f'\u2705 Updated: {last_date.date()} \u2192 {combined.index.max().date()}')
    print(f'   +{new_rows} new rows  |  total: {combined.shape}')
    return combined


# ── Run incremental update ────────────────────────────────────────────────────
updated = incremental_update(DATABENTO_API_KEY)